In [1]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import sphWarpCore_config as swc
from typing import Any
swc.configure(precision="float32", dim=Any) # precision: float16|half|float32|single|float64|double

import sphWarpCore as sph
from sphWarpCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from integrators.integration import *
from sphWarpCore import *
from warpPlot import *

# This library
from warpSPH import *

# The case utilities that contain all the case setup functions for the various test cases
from warpSPH.caseUtils import *

{'scalar_t': <class 'warp._src.types.float32'>, 'dim_t': typing.Any}
Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX PRO 500 Blackwell Generation Laptop GPU" (6 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/lu26029/.cache/warp/1.12.0


In [2]:
from utils import *
from warpSPH.io import *

In [3]:
directory = f'./compressed/'

simulations = os.listdir(directory)
simulations = [sim for sim in simulations if 'h5' in sim or 'hdf5' in sim]


print(f'Found {len(simulations)} simulations in {directory}')
# trajectoryFile = f'{directory}/trajectory.h5'
# configFile = f'{directory}/config.json'

Found 5 simulations in ./compressed/


In [4]:
def getTitleString(trajectory, schemeConfig, state, time):
    caseName = trajectory.attrs['caseName']
    timeLimit = trajectory.attrs['timeLimit']
    simulationDt = trajectory.attrs['original_dt']
    exportDt = trajectory.attrs['exportInterval']
    L = trajectory.attrs['L']
    W = trajectory.attrs['W']
    obstacleType = trajectory.attrs['obstacleType']
    obstacleActive = trajectory.attrs['obstacleActive']
    aoa = trajectory.attrs['aoa']
    nx = trajectory.attrs['nx']
    n_h = trajectory.attrs['n_h']
    fixedSoundSpeed = schemeConfig.fluid.fixedSoundSpeed

    caseText = f'{caseName}'
    timeText = f't = {time:.4g}/{timeLimit:.4g} | dt = {simulationDt:.4g} / export dt = {exportDt:.4g}'
    particleText = f'particles = {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary | nx = {nx} | n_h = {n_h}'
    domainText = f'L = {L}, W = {W}'
    obstacleText = f'obstacle: {obstacleType}, aoa: {aoa}' if obstacleActive else 'no obstacle'
    stateText = f'v_max = {state.velocities.max().cpu().item():.4g} (c0 = {fixedSoundSpeed:.4g}), rho_max = {state.densities.max().cpu().item():.4g}, rho_min = {state.densities.min().cpu().item():.4g}'


    titleString = f'{caseText} | {timeText} | {particleText}\n{domainText} | {obstacleText} | {stateText}'
    return titleString

markerSize = 8
velocityPlot = PlottingOptions(
    colorMap = UniformColorMap.viridis,
    markerSize = markerSize,
    midPoint = 0.0,
    quantityScaling = PlotScaling.Linear,
    mapping = Mapping.L2Norm,
    plotTitle = "Particle Velocity Magnitude",
    plotTitleGap = 0.08,
    boundaryVisualization = VisualizeOptions.Visualize,
    # gridVisualization = GridVisualization(
    #     resolution = 1024,
    #     streamLines = True,
    # ),

    # vMin=1e-10,
    vMin = 0.0,
    vMax = 2.7,
)
densityPlot = PlottingOptions(
    colorMap = DivergingColorMap.RdBu,
    flipColorMap=True,
    markerSize = markerSize,
    midPoint = 1.0,
    quantityScaling = PlotScaling.Linear,
    plotTitle = "Particle Density",
    # vMin = 0.95,
    # vMax = 1.05,
    plotTitleGap = 0.08,
    # gridVisualization = GridVisualization(
    #     resolution = 512,
    # ),
)
UIDPlot = PlottingOptions(
    colorMap = CyclicColorMap.twilight,
    # flipColorMap=True,
    markerSize = markerSize,
    # midPoint = 1.0,
    quantityScaling = PlotScaling.Linear,
    plotTitle = f"Particle IDs",
    boundaryVisualization = VisualizeOptions.Passive,
    # vMin = 0.95,
    # vMax = 1.05
    plotTitleGap = 0.08,
    # gridVisualization = GridVisualization(
    #     resolution = 512,
    # ),
    plotDomain = True
)


In [5]:
import h5py
import json

print(f'Found {len(simulations)} simulations in {directory}')
file = simulations[-1]

trajectoryFile = f'{directory}/{file}'
print(f'Loading trajectory from {trajectoryFile}')

# loadedConfig = json.load(open(configFile, 'r'))

def loadTrajectory(trajectoryFile):

    trajectory = h5py.File(trajectoryFile, 'r')
    numStates = trajectory['positions'].shape[0] if not isinstance(trajectory['positions'], h5py.Group) else len(trajectory['positions'].keys())
    numFluidParticles = trajectory['positions'].shape[1] if not isinstance(trajectory['positions'], h5py.Group) else len(trajectory['positions'][f'frame_{0:05d}'][:])
    print(f'Loaded trajectory with {numStates} states and {numFluidParticles} fluid particles from {trajectoryFile}')

    restoredConfig = restoreConfig_from_h5(trajectory['config'])

    scheme = restoredConfig['scheme']
    schemeEnum = schemeNameToSimulationScheme(scheme)

    SimulationSystem, SimulationState, SimulationConfig, SimulationUpdate, fn, export_fn, import_fn = buildScheme(schemeEnum)
    config, schemeConfig = dictToConfig(restoredConfig['config']), import_fn(restoredConfig['schemeConfig'])

    device = torch.device(config.device)
    kinds = torch.tensor(trajectory['combinedKinds'][:], dtype=torch.int32)

    simulationState = WeaklyCompressibleState(
        positions = torch.tensor(trajectory['combinedPositions'][:], dtype=torch.float32, device = device),
        supports = torch.tensor(trajectory['combinedSupports'][:], dtype=torch.float32, device = device),
        masses = torch.tensor(trajectory['combinedMasses'][:], dtype=torch.float32, device = device),
        densities = torch.tensor(trajectory['combinedDensities'][:], dtype=torch.float32, device = device),
        velocities = torch.tensor(trajectory['combinedVelocities'][:], dtype=torch.float32, device = device),

        pressures = None,
        soundspeeds = None,

        kinds = torch.tensor(trajectory['combinedKinds'][:], dtype=torch.int32, device = device),
        materials = torch.tensor(trajectory['combinedMaterials'][:], dtype=torch.int32, device = device),
        UIDs = torch.tensor(trajectory['combinedUIDs'][:], dtype=torch.int64, device = device),
        UIDcounter = int(trajectory['combinedPositions'][:].shape[0]),

        ghostIndices = torch.tensor(trajectory['combinedGhostIndices'][:], dtype=torch.int32, device = device),
        ghostOffsets = torch.tensor(trajectory['combinedGhostOffsets'][:], dtype=torch.float32, device = device)
    )

    return trajectory, config, schemeConfig, simulationState, device


Found 5 simulations in ./compressed/
Loading trajectory from ./compressed//trajectory_kolmogorov_2026-07-30_14-50-49_128_4_2.0_2.0_obstacle_0.5_0_0.hdf5


In [6]:
trajectory, config, schemeConfig, initialState, device = loadTrajectory(trajectoryFile)


Loaded trajectory with 500 states and 17196 fluid particles from ./compressed//trajectory_kolmogorov_2026-07-30_14-50-49_128_4_2.0_2.0_obstacle_0.5_0_0.hdf5


In [7]:
def loadFrame(trajectory, frame, initialState):
    # the data could either be stored directly as a dataset or as a group of datasets, depending on the size of the data and the HDF5 file structure. This function handles both cases.
    if not isinstance(trajectory['positions'], h5py.Group):
        # print()
        positions = trajectory['positions'][frame][:]
        densities = trajectory['densities'][frame][:]
        velocities = trajectory['velocities'][frame][:]
        time = trajectory['times'][frame].item()
    else:
        frame_keys = list(trajectory['positions'].keys())
        current_frame_key = frame_keys[frame]
        print(f'Loading frame {frame} from group {current_frame_key}')
        positions = trajectory['positions'][current_frame_key][:]
        densities = trajectory['densities'][current_frame_key][:]
        velocities = trajectory['velocities'][current_frame_key][:]
        time = trajectory['times'][current_frame_key][:].item()
    newState = WeaklyCompressibleState(
        positions = torch.tensor(positions, dtype=torch.float32, device = device),
        supports = initialState.supports,
        masses = initialState.masses,
        densities = torch.tensor(densities, dtype=torch.float32, device = device),
        velocities = torch.tensor(velocities, dtype=torch.float32, device = device),

        pressures = None,
        soundspeeds = None,

        kinds = initialState.kinds,
        materials = initialState.materials,
        UIDs = initialState.UIDs,
        UIDcounter = initialState.UIDcounter,

        ghostIndices = initialState.ghostIndices,
        ghostOffsets = initialState.ghostOffsets
    )
    return newState, time

def update_frame(frame):
    state, time = loadFrame(trajectory, frame, initialState)
    titleString = getTitleString(trajectory, schemeConfig, state, time)
    plotter.updateQuantities(
        {
            "A": state.UIDs
        },
        newParticleState = state,
        newDomain = config.domain
    )
    plotter.updateTitle(titleString)
def updateTrajectory(trajectoryFile):
    global trajectory, config, schemeConfig, initialState, device

    old_frameCount = trajectory['positions'].shape[0] if not isinstance(trajectory['positions'], h5py.Group) else len(trajectory['positions'].keys())
    
    trajectory, config, schemeConfig, initialState, device = loadTrajectory(trajectoryFile)
    new_frameCount = trajectory['positions'].shape[0] if not isinstance(trajectory['positions'], h5py.Group) else len(trajectory['positions'].keys())

    frame_slider.max = new_frameCount - 1
    if new_frameCount != old_frameCount:
        # print(f'Updated trajectory from {old_frameCount} frames to {new_frameCount} frames.')
        frame_slider.value = frame_slider.value % new_frameCount
    update_frame(frame_slider.value)

In [8]:
state, time = loadFrame(trajectory, 10, initialState)
titleString = getTitleString(trajectory, schemeConfig, state, time)

plotter = visualize(
    particleState = state,
    domain = config.domain,
    quantities = {
        # "A": state.velocities,
        "A": state.UIDs
    },
    plotOptions = {
        # "A": velocityPlot,
        "A": UIDPlot
    },
    figTitle = titleString,
    mosaic = 'A',
    figsize= (12, 9),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

from ipywidgets import widgets
frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(trajectory['positions']) - 1,
    step=1,
    description='Frame:',
    continuous_update=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

frame_slider.observe(lambda change: update_frame(change['new']), names='value')

file_dropdown = widgets.Dropdown(
    options=simulations,
    value=simulations[0],
    description='Select Simulation:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

file_dropdown.observe(lambda change: updateTrajectory(f'{directory}/{change["new"]}'), names='value')

display(frame_slider)
display(file_dropdown)

Loading frame 10 from group frame_00020


RFBOutputContext()

IntSlider(value=0, description='Frame:', layout=Layout(width='80%'), max=499, style=SliderStyle(description_wi…

Dropdown(description='Select Simulation:', layout=Layout(width='80%'), options=('trajectory_kolmogorov_2026-07…

In [11]:
testDomain = buildDomainDescription(0.5, 2, True, device)
# plotter.updateDomain(testDomain)
plotter.updateQuantities(
        {
            "A": state.UIDs
        },
        newDomain = config.domain,
        newParticleState = state
)


In [17]:
schemeConfig.fluid.fixedSoundSpeed

26.98631979252399

In [18]:
runningState = compressibleSystem.initializeNewState()

In [19]:

markerSize = 8
plotter = visualize(
    particleState = runningState.state,
    domain = config.domain,
    quantities = {
        "A": runningState.state.velocities,
        "B":runningState.state.UIDs
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            mapping = Mapping.L2Norm,
            plotTitle = "velocities",
            plotTitleGap = 0.08,
            boundaryVisualization = VisualizeOptions.Passive,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
            # vMin=1e-10
            # vMin = 0.0,
            # vMax = schemeConfig.fluid.fixedSoundSpeed * 0.1
        ),
        "B": PlottingOptions(
            colorMap = CyclicColorMap.twilight,
            # flipColorMap=True,
            markerSize = markerSize,
            # midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "UIDs",
            # vMin = 0.95,
            # vMax = 1.05
            plotTitleGap = 0.08,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        )
    },
    figTitle = "Taylor Green Vortex",
    mosaic = 'AB',
    figsize= (14,5),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)


RFBOutputContext()

In [22]:
integrator = getIntegrator(config.integrationScheme)
config.dim = 2

In [25]:
t_limit = 1.28
nSteps = int(t_limit / config.dt)

runningState = compressibleSystem.initializeNewState()
runningState.adjacency = None
# schemeConfig.rigidBodies[0].linearVelocity = 0.0

kes = []
priorStep = None
for i in (tq := tqdm(range(nSteps), leave = False)):
    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    result = integrator.function(
        state = runningState,
        f = fn,
        dt = config.dt,  
        config = config,
        schemeConfig = schemeConfig,
        verbose = False,
        # priorStep = priorStep
    )
    kes.append(torch.sum(0.5 * result.state.state.masses * torch.sum(result.state.state.velocities**2, dim=1)))
    # print('max_vel:', torch.linalg.norm(result.state.state.velocities, dim = -1).max())
    end.record()
    torch.cuda.synchronize()
    priorStep = result.stages[-1]
    timing = begin.elapsed_time(end)

    runningState = result.state
    t = runningState.t
    # schemeConfig.rigidBodies[0].linearVelocity = 0.5 * torch.cos(t * np.pi * 2)
    # linearVelocity = 0.5 * torch.cos(t * np.pi * 2)

    currentState = runningState.state
    # print(f'-' * 80)
    # print(f'Fluid density stats: min={currentState.densities[currentState.kinds == 0].min().item()}, max={currentState.densities[currentState.kinds == 0].max().item()}, mean={currentState.densities[currentState.kinds == 0].mean().item()}')
    # print(f'Boundary density stats: min={currentState.densities[currentState.kinds == 1].min().item()}, max={currentState.densities[currentState.kinds == 1].max().item()}, mean={currentState.densities[currentState.kinds == 1].mean().item()}')
    if i % 20 == 0 :
        # densities = computeDensities(runningState.state, config, schemeConfig, None)

        plotter.updateQuantities(
            {
                "A": runningState.state.velocities,
                # "B": runningState.state.densities,
                "B": runningState.state.UIDs
            },
            newParticleState = runningState.state,
        )
        # plotter.export(f'{imagePath}/frame_{i:05d}.png', dpi = 300)
    # break
        
    maxVel = torch.linalg.norm(runningState.state.velocities, dim = -1).max()
    tq.set_description(f"Step {i+1}/{nSteps}, time: {(i+1)*config.dt:8.4g}/{t_limit:8.4g} | max vel: {maxVel:.3g} | iter time: {timing:.3f} ms")
    # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
    # break
    if torch.any(torch.isnan(runningState.state.velocities)):
        print("NaN detected in velocities, stopping simulation.")
        break



  0%|          | 0/6400 [00:00<?, ?it/s]

In [15]:
config.kernel

<KernelFunctions.Wendland4: 1>

In [ ]:
print(trajectory.attrs.keys())

In [ ]:
caseName = trajectory.attrs['caseName']
timeLimit = trajectory.attrs['timeLimit']
# dt = trajectory['states']['frame_00001'].attrs['time'] - trajectory['states']['frame_00000'].attrs['time']
nx = trajectory.attrs['nx']
n_h = trajectory.attrs['n_h']
L = trajectory.attrs['L']
W = trajectory.attrs['W']
obstacleType = trajectory.attrs['obstacleType']
aoa = trajectory.attrs['aoa']
obstacleActive = trajectory.attrs['obstacleActive']

dt = loadedConfig['config']['dt']
fixedSoundSpeed = loadedConfig['schemeConfig']['fixedSoundSpeed']

device = torch.device('cpu')
dtype = torch.float32

# dx = loadedConfig['config']['dx']
# band = trajectory.attrs['band']
dim = loadedConfig['config']['dim']
domain = buildDomainDescription(L, dim, True, device, dtype)
domain.min = torch.tensor(loadedConfig['config']['domain']['min'], device=device, dtype=dtype)
domain.max = torch.tensor(loadedConfig['config']['domain']['max'], device=device, dtype=dtype)

In [ ]:
markerSize = 6
plotWidth = 28
plotHeight = 10

In [ ]:
stateIndex = 0
state, time = load_state(stateIndex, trajectory)

caseText = f'{caseName}'
timeText = f't = {time:.4g}/{timeLimit:.4g} | dt = {dt:.4g}'
particleText = f'particles = {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary | nx = {nx} | n_h = {n_h}'
domainText = f'L = {L}, W = {W}'
obstacleText = f'obstacle: {obstacleType}, aoa: {aoa}' if obstacleActive else 'no obstacle'
stateText = f'v_max = {state.velocities.max().cpu().item():.4g} (c0 = {fixedSoundSpeed:.4g}), rho_max = {state.densities.max().cpu().item():.4g}, rho_min = {state.densities.min().cpu().item():.4g}'
timingText = f'iter time: {0.00:.3f} ms'

titleString = f'{caseText} | {timeText} | {particleText} | {domainText} | {obstacleText} | {stateText} | {timingText}'

from ipywidgets import widgets

def update_frame(frame_index):
    global state, time
    state, time = load_state(frame_index, trajectory)
    plotter.updateQuantities({
        # "A": state.velocities,
        "A": state.UIDs
    }, newParticleState=state)
    caseText = f'{caseName}'
    timeText = f't = {time:.4g}/{timeLimit:.4g} | dt = {dt:.4g}'
    particleText = f'particles = {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary | nx = {nx} | n_h = {n_h}'
    domainText = f'L = {L}, W = {W}'
    obstacleText = f'obstacle: {obstacleType}, aoa: {aoa}' if obstacleActive else 'no obstacle'
    stateText = f'v_max = {state.velocities.max().cpu().item():.4g} (c0 = {fixedSoundSpeed:.4g}), rho_max = {state.densities.max().cpu().item():.4g}, rho_min = {state.densities.min().cpu().item():.4g}'
    timingText = f'iter time: {0.00:.3f} ms'

    titleString = f'{caseText} | {timeText} | {particleText} | {domainText} | {obstacleText} | {stateText} | {timingText}'

    plotter.updateTitle(titleString)


markerSize = 12
velocityPlot = PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            mapping = Mapping.L2Norm,
            plotTitle = "Particle Velocity Magnitude",
            plotTitleGap = 0.08,
            boundaryVisualization = VisualizeOptions.Visualize,
            # gridVisualization = GridVisualization(
            #     resolution = 1024,
            #     streamLines = True,
            # ),

            # vMin=1e-10,
            vMin = 0.0,
            vMax = fixedSoundSpeed * 0.1,
        )
densityPlot = PlottingOptions(
            colorMap = DivergingColorMap.RdBu,
            flipColorMap=True,
            markerSize = markerSize,
            midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "Particle Density",
            # vMin = 0.95,
            # vMax = 1.05,
            plotTitleGap = 0.08,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        )
UIDPlot = PlottingOptions(
            colorMap = CyclicColorMap.twilight,
            # flipColorMap=True,
            markerSize = markerSize,
            # midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = f"Particle IDs ({caseName},  {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary particles)",
            boundaryVisualization = VisualizeOptions.Passive,
            # vMin = 0.95,
            # vMax = 1.05
            plotTitleGap = 0.08,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        )


In [ ]:

plotter = visualize(
    particleState = state,
    domain = domain,
    quantities = {
        # "A": state.velocities,
        "A": state.UIDs
    },
    plotOptions = {
        # "A": velocityPlot,
        "A": UIDPlot
    },
    figTitle = titleString,
    mosaic = 'A',
    figsize= (plotWidth, plotHeight),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

plotter.updateTitle(titleString)


frame_slider = widgets.IntSlider(
    value=500,
    min=0,
    max=numStates-1,
    step=1,
    description='Frame:',
    continuous_update=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

frame_slider.observe(lambda change: update_frame(change['new']), names='value')
# update_frame(frame_slider.value)

export_button = widgets.Button(
    description='Export Current Frame',
    button_style='success',
    tooltip='Export the current frame as an image',
    icon='download'
)
export_button.on_click(lambda b: plotter.export(f'{caseName}_frame_{frame_slider.value:05d}.png'))


display(frame_slider)
display(export_button)


In [ ]:
uMax = torch.linalg.norm(state.velocities, dim=1).max().cpu().item()
dx = state.masses[state.kinds == 0].mean().cpu().item() ** (1.0 / dim)  # approximate particle spacing based on mass and density
cfl = uMax * dt / dx
print(f"Max velocity: {uMax:.4g}, dx: {dx:.4g}, dt: {dt:.4g}, CFL number: {cfl:.4g}")

targetCFL = 1.0
maxDt = targetCFL * dx / uMax
print(f"Target CFL: {targetCFL:.4g}, Max dt for target CFL: {maxDt:.4g}")
dtRatio = dt / maxDt
print(f"dt ratio: {dtRatio:.4g} (dt / maxDt for target CFL)")
print(f'Maximum Coarse Graining Factor: {1/dtRatio:.4g} (1 / dtRatio for target CFL)')